In [ ]:
import xarray as xr
import xskillscore
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cartopy.crs as ccrs
import cartopy.io.shapereader
from scipy import stats
import re

os.makedirs("figures/RPSS drivers", exist_ok=True)


## Load and align data

In [ ]:
obs_unet = xr.open_dataarray('outputs/Full Period/GEFS_IMD/unet_ytest_raw_wk3-4.nc')
obs_ohe  = xr.open_dataarray('outputs/Full Period/GEFS_IMD/unet_y_test_wk3-4.nc')
preds    = xr.open_dataarray('outputs/Full Period/GEFS_IMD/unet_predictions_wk3-4.nc')
preds_elr = xr.open_dataarray('outputs/Full Period/GEFS_IMD/ELR_predictions_wk3-4.nc')

# obs_unet: (bootstrap, T=396, Y, X) — raw labels, real datetime T coords
# obs_ohe:  (bootstrap, T=66,  Y, X, category) — one-hot, integer T index
# preds:    (bootstrap, T=66,  Y, X, category) — probabilities, integer T index

# Recover real datetime T coords for obs_ohe / preds from obs_unet
valid_T_per_bootstrap = {}
for b in obs_unet.bootstrap.values:
    da = obs_unet.sel(bootstrap=b)
    valid = ~da.isnull().all(dim=("Y", "X"))
    valid_T_per_bootstrap[b] = da['T'][valid].values

# Assign real T coords and average over bootstraps
obs_ohe_list, preds_list = [], []
for b in range(len(obs_unet.bootstrap)):
    valid_T = valid_T_per_bootstrap[b]
    obs_ohe_list.append(obs_ohe.sel(bootstrap=b).assign_coords(T=valid_T))
    preds_list.append(preds.sel(bootstrap=b).assign_coords(T=valid_T))

obs_oheM  = xr.concat(obs_ohe_list, dim="bootstrap").mean(dim="bootstrap")   # (T, Y, X, cat)
preds_M   = xr.concat(preds_list,   dim="bootstrap").mean(dim="bootstrap")   # (T, Y, X, cat)
obs_unetM = obs_unet.mean(dim="bootstrap")                                    # (T, Y, X)

# Remove all-NaN time steps
valid_t = ~obs_oheM.isnull().all(dim=("Y","X","category"))
obs_oheM  = obs_oheM.sel(T=valid_t)
preds_M   = preds_M.sel(T=valid_t)

# Assign real lat/lon coords (obs_ohe uses integer indices, obs_unet has real coords)
real_X = obs_unet.X
real_Y = obs_unet.Y
obs_oheM  = obs_oheM.assign_coords(X=real_X, Y=real_Y)
preds_M   = preds_M.assign_coords(X=real_X, Y=real_Y)

land_mask = preds_elr.isel(bootstrap=0, category=0).isnull().all(dim='T')
land_mask = land_mask.assign_coords(X=real_X, Y=real_Y)

print("obs_oheM shape:", obs_oheM.shape)
print("preds_M  shape:", preds_M.shape)
print("T range:", str(obs_oheM.T.values[0])[:10], "to", str(obs_oheM.T.values[-1])[:10])


## RPSS — aggregated formulation (consistent with `tune_GEFS_full`)

`xskillscore.rps` with `dim='T'` sums RPS over all dates before dividing:
$$\text{RPSS}(\mathbf{x}) = 1 - \frac{\sum_t \text{RPS}_t(\mathbf{x})}{\sum_t \text{RPS}^{\text{clim}}_t(\mathbf{x})}$$


In [ ]:
# obs_ohe is already one-hot (0/1); xskillscore expects (T, Y, X, category)
obs_r  = obs_oheM.transpose('T','Y','X','category')   # one-hot obs
fcast  = preds_M.transpose('T','Y','X','category')    # model forecast

# Climatological forecast: 1/3 per category, same shape as fcast
climo  = xr.full_like(fcast, 1/3)

rps_model = xskillscore.rps(obs_r, fcast,  dim='T', category_edges=None, input_distributions='p')
rps_climo = xskillscore.rps(obs_r, climo,  dim='T', category_edges=None, input_distributions='p')

rpss = (1 - rps_model / rps_climo).where(~land_mask).assign_coords(X=real_X, Y=real_Y)

print("RPSS range:", float(rpss.min().values), "to", float(rpss.max().values))


In [ ]:
def plot_map(da, title="", cmap="RdBu_r", vmin=-0.2, vmax=0.2):
    fig, ax = plt.subplots(figsize=(8,6), subplot_kw={'projection': ccrs.PlateCarree()})
    im = da.plot(ax=ax, transform=ccrs.PlateCarree(), cmap=cmap,
                 vmin=vmin, vmax=vmax, add_colorbar=False)
    ax.coastlines()
    reader = cartopy.io.shapereader.Reader('shapes/indian_borders.shp')
    ax.add_geometries(reader.geometries(), ccrs.PlateCarree(),
                      facecolor='none', edgecolor='black')
    ax.set_title(title, fontsize=12, fontweight="bold", pad=10)
    gl = ax.gridlines(draw_labels=True, linestyle="--", linewidth=0.5, alpha=0.7)
    gl.top_labels = False; gl.right_labels = False
    plt.colorbar(im, ax=ax, orientation="vertical", pad=0.02, fraction=0.05)
    plt.show()

plot_map(rpss, title="GEFSv12 wk3-4 RPSS (aggregated)")


# MJO

In [ ]:
mjo = pd.read_csv(
    "download/mjo.txt", skiprows=2, sep=r'\s+', header=None,
    names=["year","month","day","RMM1","RMM2","phase","amplitude","method","extra"],
    na_values=[1e35, 999]
).drop(columns="extra", errors="ignore")
mjo["date"] = pd.to_datetime(dict(year=mjo.year, month=mjo.month, day=mjo.day), errors="coerce")
mjo = mjo[(mjo['date'] >= '1980-01-01') & (mjo['date'] <= '2025-12-31')].set_index('date')

df_mjo_roll = mjo[['RMM1','RMM2']].rolling(window=15, center=True, min_periods=1).mean()
mjo_xr = df_mjo_roll.to_xarray()

rpss_dates = pd.to_datetime(obs_oheM['T'].values)
rmm1 = mjo_xr['RMM1'].sel(date=rpss_dates, method='nearest').values
rmm2 = mjo_xr['RMM2'].sel(date=rpss_dates, method='nearest').values
phase_num = (((np.arctan2(rmm2, rmm1) + 2*np.pi) % (2*np.pi)) // (np.pi/4) + 1).astype(int)
phase_da  = xr.DataArray(phase_num, dims=('T',), coords={'T': obs_oheM['T']})
print("Phase counts:", {int(p): int((phase_da==p).sum()) for p in range(1,9)})


## RPSS per MJO phase

In [ ]:
rpss_mjo = {}
for phase in range(1, 9):
    mask_t = (phase_da == phase).values
    if mask_t.sum() < 3:
        continue
    obs_ph  = obs_r.isel(T=mask_t)
    fcast_ph = fcast.isel(T=mask_t)
    climo_ph = climo.isel(T=mask_t)
    rps_m = xskillscore.rps(obs_ph, fcast_ph, dim='T', category_edges=None, input_distributions='p')
    rps_c = xskillscore.rps(obs_ph, climo_ph, dim='T', category_edges=None, input_distributions='p')
    rpss_mjo[phase] = (1 - rps_m / rps_c).where(~land_mask)

print("Phases computed:", list(rpss_mjo.keys()))


In [ ]:
def plot_mjo_phases(rpss_dict, demean=True, rpss_ref=None,
                   vmin=-0.2, vmax=0.2, cmap='bwr', fname=None):
    if rpss_ref is None:
        rpss_ref = rpss
    fig, axes = plt.subplots(nrows=2, ncols=4, figsize=(18,8),
                             subplot_kw={"projection": ccrs.PlateCarree()})
    axes = axes.flatten(); mappable = None

    for i, phase in enumerate(range(1,9)):
        ax  = axes[i]
        val = rpss_dict[phase]
        if demean:
            val = val - rpss_ref
        im = val.plot(ax=ax, transform=ccrs.PlateCarree(),
                      cmap=cmap, vmin=vmin, vmax=vmax, add_colorbar=False)
        if mappable is None: mappable = im
        ax.coastlines()
        reader = cartopy.io.shapereader.Reader('shapes/indian_borders.shp')
        ax.add_geometries(reader.geometries(), ccrs.PlateCarree(),
                          facecolor='none', edgecolor='black', linewidth=0.8)
        ax.set_title(f"MJO Phase {phase}", fontsize=14, fontweight="bold", pad=6)
        gl = ax.gridlines(draw_labels=True, linestyle="--", linewidth=0.4, alpha=0.6)
        gl.top_labels = False; gl.right_labels = False
        if i % 4 != 0: gl.left_labels = False
        if i < 4:      gl.bottom_labels = False
        gl.xlabel_style = {"size":10}; gl.ylabel_style = {"size":10}

    cbar = fig.colorbar(mappable, ax=axes, orientation="horizontal", fraction=0.05, pad=0.08)
    cbar.set_label("RPSS deviation from overall" if demean else "RPSS", fontsize=14)
    if fname:
        plt.savefig(f"figures/RPSS drivers/{fname}", dpi=300, bbox_inches="tight")
    plt.show()

plot_mjo_phases(rpss_mjo, demean=False, vmin=-0.2, vmax=0.2, fname="RPSS_MJO_plain.pdf")


## MJO composite maps with stippling (phase vs all, p < 0.10)

# ENSO

In [ ]:
records = []
pattern = re.compile(r"[+-]?\d+\.\d+")
with open("download/sst_weekly.txt") as f:
    for line in f:
        if re.match(r"\s*\d{2}[A-Z]{3}\d{4}", line):
            date = line.split()[0]
            nums = list(map(float, pattern.findall(line)))
            if len(nums) == 8:
                records.append([date] + nums)

cols = ["Week","Nino12_SST","Nino12_SSTA","Nino3_SST","Nino3_SSTA",
        "Nino34_SST","Nino34_SSTA","Nino4_SST","Nino4_SSTA"]
nino = pd.DataFrame(records, columns=cols)
nino["Week"] = pd.to_datetime(nino["Week"], format="%d%b%Y")
nino = nino.set_index("Week").rolling(window=2, center=True, min_periods=1).mean()
nino_xr = nino.to_xarray()

nino4_vals  = nino_xr["Nino4_SSTA"].sel(Week=rpss_dates, method="nearest").values
nino4_vals = nino_xr["Nino4_SSTA"].sel(Week=rpss_dates, method="nearest").values

enso4_da  = xr.DataArray(xr.where(xr.DataArray(nino4_vals,  dims='T') >=  0.5, 1,
                          xr.where(xr.DataArray(nino4_vals,  dims='T') <= -0.5, -1, 0)).values,
                         dims=('T',), coords={'T': obs_oheM['T']})
enso4_da = xr.DataArray(xr.where(xr.DataArray(nino4_vals, dims='T') >=  0.5, 1,
                          xr.where(xr.DataArray(nino4_vals, dims='T') <= -0.5, -1, 0)).values,
                         dims=('T',), coords={'T': obs_oheM['T']})

print("ENSO4  counts:", {int(v): int((enso4_da ==v).sum()) for v in [-1,0,1]})
print("ENSO4 counts:", {int(v): int((enso4_da==v).sum()) for v in [-1,0,1]})


## RPSS per ENSO mode + composite stippling maps

In [ ]:
def rpss_by_enso(mode_da):
    out = {}
    for mode in [-1, 0, 1]:
        mask_t = (mode_da == mode).values
        if mask_t.sum() < 3: continue
        rps_m = xskillscore.rps(obs_r.isel(T=mask_t), fcast.isel(T=mask_t),
                                dim='T', category_edges=None, input_distributions='p')
        rps_c = xskillscore.rps(obs_r.isel(T=mask_t), climo.isel(T=mask_t),
                                dim='T', category_edges=None, input_distributions='p')
        out[mode] = (1 - rps_m / rps_c).where(~land_mask)
    return out

rpss_enso4  = rpss_by_enso(enso4_da)
rpss_enso4 = rpss_by_enso(enso4_da)
print("Done.")


## Plain RPSS per ENSO mode (Niño4 and Niño3.4)

In [ ]:
def plot_enso_plain(rpss_enso, titles, fname):
    fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(13, 5),
                             subplot_kw={"projection": ccrs.PlateCarree()})
    mappable = None

    for i, mode in enumerate([-1, 0, 1]):
        ax  = axes[i]
        val = rpss_enso[mode]

        im = val.plot(ax=ax, transform=ccrs.PlateCarree(),
                      cmap='bwr', vmin=-0.2, vmax=0.2, add_colorbar=False)
        if mappable is None:
            mappable = im

        ax.coastlines()
        reader = cartopy.io.shapereader.Reader('shapes/indian_borders.shp')
        ax.add_geometries(reader.geometries(), ccrs.PlateCarree(),
                          facecolor='none', edgecolor='black', linewidth=0.8)
        ax.set_title(titles[mode], fontsize=13, fontweight="bold", pad=6)

        gl = ax.gridlines(draw_labels=True, linestyle="--", linewidth=0.4, alpha=0.6)
        gl.top_labels = False; gl.right_labels = False
        if i == 1: gl.left_labels = False
        gl.xlabel_style = {"size": 10}; gl.ylabel_style = {"size": 10}

    cbar = fig.colorbar(mappable, ax=axes, orientation="horizontal",
                        fraction=0.05, pad=0.12)
    cbar.set_label("RPSS", fontsize=13)
    plt.savefig(f"figures/RPSS drivers/{fname}", dpi=300, bbox_inches="tight")
    plt.show()

nino4_titles  = {-1: r"La Niña (Niño4 $\leq -0.5°$C)",  0: "Neutral",
                  1: r"El Niño (Niño4 $\geq +0.5°$C)"}
nino4_titles = {-1: r"La Niña (Niño4 $\leq -0.5°$C)", 0: "Neutral",
                  1: r"El Niño (Niño4 $\geq +0.5°$C)"}

plot_enso_plain(rpss_enso4,  nino4_titles,  "RPSS_ENSO4_plain.pdf")
plot_enso_plain(rpss_enso4, nino4_titles, "RPSS_ENSO4_plain.pdf")


## Box plots with significance stars

# Combined MJO phase × ENSO mode stratification

In [ ]:
# ── Assign MJO amplitude to each date ────────────────────────────
amplitude = np.sqrt(rmm1**2 + rmm2**2)
amplitude_da = xr.DataArray(amplitude, dims=('T',), coords={'T': obs_oheM['T']})

# Active MJO: amplitude >= 1
active_da = amplitude_da >= 1.0
print(f"Active MJO dates: {int(active_da.sum())} / {len(active_da)}")
print(f"Phase counts (active only):")
for ph in range(1, 9):
    n = int(((phase_da == ph) & active_da).sum())
    print(f"  Phase {ph}: {n}")
